In [1]:
# setup
import pandas as pd
import numpy as np

df = pd.read_csv("Filemaker_CLEAN.csv")

df = df[[
    "uniqname",
    "experience_type",
    "year",
    "term_season"
]]

df = df.dropna()
df["experience_type"] = df["experience_type"].str.strip()

# Remove exact duplicates
df = df.drop_duplicates()

In [2]:
# Time ordering
season_map = {"WN": 1, "SP": 2, "SU": 3, "FA": 4}
df["season_num"] = df["term_season"].map(season_map)

df = df.sort_values(["uniqname", "year", "season_num"])

In [3]:
# Build sequences

# Raw sequences (keep repeats)
raw_sequences = df.groupby("uniqname")["experience_type"].apply(list)

# Collapse consecutive duplicates
def remove_consecutive_duplicates(seq):
    return [seq[i] for i in range(len(seq)) if i == 0 or seq[i] != seq[i-1]]

collapsed_sequences = raw_sequences.apply(remove_consecutive_duplicates)

# First Touch Analysis + Pathways

In [4]:

# RAW
first_touch_raw = raw_sequences.apply(lambda x: x[0] if len(x) > 0 else None)

print("\n=== First Touch (RAW) ===")
print(first_touch_raw.value_counts(normalize=True))


=== First Touch (RAW) ===
experience_type
Event                     0.574030
Course                    0.254065
Counseling Appointment    0.094949
Funding                   0.076956
Name: proportion, dtype: float64


In [5]:
# Collapsed
first_touch_collapsed = collapsed_sequences.apply(lambda x: x[0] if len(x) > 0 else None)

print("\n=== First Touch (COLLAPSED) ===")
print(first_touch_collapsed.value_counts(normalize=True))


=== First Touch (COLLAPSED) ===
experience_type
Event                     0.574030
Course                    0.254065
Counseling Appointment    0.094949
Funding                   0.076956
Name: proportion, dtype: float64


In [17]:
# Transitions RAW
raw_transitions = []

for seq in raw_sequences:
    for i in range(len(seq) - 1):
        raw_transitions.append((seq[i], seq[i+1]))

raw_transitions_df = pd.DataFrame(raw_transitions, columns=["current", "next"])

raw_counts = (
    raw_transitions_df.groupby(["current", "next"])
    .size()
    .reset_index(name="count")
)

raw_counts["total"] = raw_counts.groupby("current")["count"].transform("sum")
raw_counts["prob"] = raw_counts["count"] / raw_counts["total"]

# Baseline
baseline = df["experience_type"].value_counts(normalize=True)
raw_counts["baseline"] = raw_counts["next"].map(baseline)
raw_counts["lift"] = raw_counts["prob"] / raw_counts["baseline"]


print("\n=== RAW TRANSITIONS ===")
print(raw_counts.sort_values("count", ascending=False).head(20))


=== RAW TRANSITIONS ===
                   current                    next  count  total      prob  \
10                   Event                   Event   1110   2714  0.408990   
9                    Event                  Course    713   2714  0.262712   
2   Counseling Appointment                   Event    589   1167  0.504713   
5                   Course                  Course    560   1643  0.340840   
6                   Course                   Event    541   1643  0.329276   
8                    Event  Counseling Appointment    455   2714  0.167649   
11                   Event                 Funding    436   2714  0.160648   
4                   Course  Counseling Appointment    395   1643  0.240414   
14                 Funding                   Event    275    661  0.416036   
1   Counseling Appointment                  Course    248   1167  0.212511   
3   Counseling Appointment                 Funding    196   1167  0.167952   
12                 Funding  Counseling 

In [14]:
# Transitions COLLAPSED
collapsed_transitions = []

for seq in collapsed_sequences:
    for i in range(len(seq) - 1):
        collapsed_transitions.append((seq[i], seq[i+1]))

collapsed_df = pd.DataFrame(collapsed_transitions, columns=["current", "next"])

collapsed_counts = (
    collapsed_df.groupby(["current", "next"])
    .size()
    .reset_index(name="count")
)

collapsed_counts["total"] = collapsed_counts.groupby("current")["count"].transform("sum")
collapsed_counts["prob"] = collapsed_counts["count"] / collapsed_counts["total"]

collapsed_counts["baseline"] = collapsed_counts["next"].map(baseline)
collapsed_counts["lift"] = collapsed_counts["prob"] / collapsed_counts["baseline"]

collapsed_counts = collapsed_counts.sort_values("lift", ascending=False).reset_index(drop=True)

print("\n=== COLLAPSED TRANSITIONS ===")
print(collapsed_counts.sort_values("lift", ascending=False).head(20))


=== COLLAPSED TRANSITIONS ===
                   current                    next  count  total      prob  \
0                   Course  Counseling Appointment    395   1083  0.364728   
1                    Event                 Funding    436   1604  0.271820   
2                  Funding  Counseling Appointment    189    581  0.325301   
3                    Event  Counseling Appointment    455   1604  0.283666   
4                    Event                  Course    713   1604  0.444514   
5   Counseling Appointment                 Funding    196   1033  0.189739   
6                   Course                 Funding    147   1083  0.135734   
7   Counseling Appointment                   Event    589   1033  0.570184   
8                   Course                   Event    541   1083  0.499538   
9                  Funding                   Event    275    581  0.473322   
10  Counseling Appointment                  Course    248   1033  0.240077   
11                 Funding       

# High Engagement Analysis

In [8]:
# Define High Engagement
engagement_counts = df.groupby("uniqname").size().rename("total_engagements")

student_df = engagement_counts.reset_index()

threshold = student_df["total_engagements"].quantile(0.75)

student_df["high_engagement"] = (student_df["total_engagements"] >= threshold).astype(int)

In [9]:
# RAW Features
raw_flags = (
    df.assign(val=1)
    .pivot_table(index="uniqname", columns="experience_type", values="val", aggfunc="max", fill_value=0)
)

features_raw = student_df.set_index("uniqname").join(raw_flags)

In [10]:
# EARLY Features
df["order"] = df.groupby("uniqname").cumcount()

early_df = df[df["order"] <= 1]

early_flags = (
    early_df.assign(val=1)
    .pivot_table(index="uniqname", columns="experience_type", values="val", aggfunc="max", fill_value=0)
)

features_early = student_df.set_index("uniqname").join(early_flags)

In [11]:
# COLLAPSED Features
collapsed_flags = pd.DataFrame(index=collapsed_sequences.index)

for exp_type in df["experience_type"].unique():
    collapsed_flags[exp_type] = collapsed_sequences.apply(lambda seq: int(exp_type in seq))

features_collapsed = student_df.set_index("uniqname").join(collapsed_flags)

In [12]:
# General Impact Function (use on each feature set)

def compute_lift(features, label=""):
    results = {}

    for col in features.columns:
        if col in ["total_engagements", "high_engagement"]:
            continue
        
        group = features.groupby(col)["high_engagement"].mean()
        
        if 0 in group and 1 in group:
            results[col] = {
                "No": group[0],
                "Yes": group[1],
                "Lift": group[1] / group[0] if group[0] > 0 else np.nan
            }

    result_df = pd.DataFrame(results).T.sort_values("Lift", ascending=False)
    
    print(f"\n=== IMPACT ({label}) ===")
    print(result_df)
    
    return result_df

In [13]:
# Run all 3 analyses
impact_raw = compute_lift(features_raw, "RAW (Ever Participated)")
impact_early = compute_lift(features_early, "EARLY (First 2 Interactions)")
impact_collapsed = compute_lift(features_collapsed, "COLLAPSED (Ever in Sequence)")


=== IMPACT (RAW (Ever Participated)) ===
                              No       Yes      Lift
Counseling Appointment  0.162687  0.743818  4.572069
Funding                 0.205858  0.610417  2.965229
Event                   0.126900  0.355171  2.798835
Course                  0.174631  0.475948  2.725448

=== IMPACT (EARLY (First 2 Interactions)) ===
                              No       Yes      Lift
Counseling Appointment  0.228052  0.633144  2.776314
Course                  0.253575  0.368349  1.452625
Event                   0.243581  0.312864  1.284432
Funding                 0.282286  0.345009  1.222196

=== IMPACT (COLLAPSED (Ever in Sequence)) ===
                              No       Yes      Lift
Counseling Appointment  0.162687  0.743818  4.572069
Funding                 0.205858  0.610417  2.965229
Event                   0.126900  0.355171  2.798835
Course                  0.174631  0.475948  2.725448
